In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
%cd /content/drive/MyDrive/TLC_project

/content/drive/MyDrive/TLC_project


In [57]:
!dvc remote remove myremote

In [58]:
!dvc remote add -d myremote /content/drive/MyDrive/TLC_DVC_Storage

Setting 'myremote' as a default remote.


In [31]:
!mkdir -p data/raw/taxi
!mkdir -p data/context
!mkdir -p src

In [32]:
!pip install dvc dvc-gdrive --quiet

In [33]:
!git init
!dvc init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/TLC_project/.git/
Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|        

In [40]:
!git config --global user.name "hiyhihi"
!git config --global user.email "huyphan1610@gmail.com"

In [38]:
!git remote add origin https://github.com/xn-dung/mining_of_massive_dataset

In [39]:
!git remote -v

origin	https://github.com/xn-dung/mining_of_massive_dataset (fetch)
origin	https://github.com/xn-dung/mining_of_massive_dataset (push)


In [41]:
!git pull origin main

remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 134 (delta 28), reused 23 (delta 23), pack-reused 97 (from 1)
Receiving objects: 100% (134/134), 250.56 KiB | 5.11 MiB/s, done.
Resolving deltas: 100% (54/54), done.
From https://github.com/xn-dung/mining_of_massive_dataset
 * branch            main       -> FETCH_HEAD
 * [new branch]      main       -> origin/main


In [42]:
!git checkout -b feature/data-update


Switched to a new branch 'feature/data-update'


In [43]:
!git branch

* feature/data-update
  master



# Download TLC parquet


In [44]:
import os
import requests
import re
import urllib.request
from tqdm import tqdm

def download_yellow_taxi_data(years, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    tlc_url = 'https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page'

    print(f"Đang lấy dữ liệu từ TLC cho các năm: {list(years)}")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    }

    response = requests.get(tlc_url, headers=headers)

    if response.status_code != 200:
        print(f"Không thể truy cập trang web TLC. Mã lỗi: {response.status_code}")
        return

    for target_year in years:
        print(f"\n--- BẮT ĐẦU TẢI NĂM {target_year} ---")

        pattern = rf'https://[^"]*yellow_tripdata_{target_year}-\d{{2}}\.parquet'
        parquet_links = list(set(re.findall(pattern, response.text)))

        if not parquet_links:
            print(f"Không tìm thấy dữ liệu Yellow Taxi cho năm {target_year}.")
            continue

        print(f"Tìm thấy {len(parquet_links)} files cho năm {target_year}.")

        for file_url in sorted(parquet_links):
            file_name = file_url.split('/')[-1]
            file_path = os.path.join(output_dir, file_name)

            if os.path.exists(file_path):
                print(f"File {file_name} đã tồn tại. Bỏ qua.")
                continue

            print(f'Đang tải {file_name}...')

            try:
                req_with_headers = urllib.request.Request(file_url, headers=headers)

                response_stream = urllib.request.urlopen(req_with_headers)
                file_size = int(response_stream.info().get('Content-Length', -1))

                with open(file_path, 'wb') as file, tqdm(
                    desc=file_name, total=file_size, unit='B', unit_scale=True, unit_divisor=1024
                ) as bar:
                    block_size = 8192
                    while True:
                        buffer = response_stream.read(block_size)
                        if not buffer:
                            break
                        file.write(buffer)
                        bar.update(len(buffer))

            except urllib.error.HTTPError as e:
                print(f"Lỗi HTTP {e.code} khi tải {file_name}: {e.reason}")
            except Exception as e:
                print(f"Lỗi không xác định khi tải {file_name}: {e}")

if __name__ == "__main__":
    TARGET_YEARS = range(2022, 2025)
    OUTPUT_DIR = "/content/drive/MyDrive/TLC_project/data/raw/taxi"
    download_yellow_taxi_data(TARGET_YEARS, OUTPUT_DIR)

Đang lấy dữ liệu từ TLC cho các năm: [2022, 2023, 2024]

--- BẮT ĐẦU TẢI NĂM 2022 ---
Tìm thấy 12 files cho năm 2022.
Đang tải yellow_tripdata_2022-01.parquet...


yellow_tripdata_2022-01.parquet: 100%|██████████| 36.4M/36.4M [00:00<00:00, 74.6MB/s]


Đang tải yellow_tripdata_2022-02.parquet...


yellow_tripdata_2022-02.parquet: 100%|██████████| 43.5M/43.5M [00:00<00:00, 60.1MB/s]


Đang tải yellow_tripdata_2022-03.parquet...


...

yellow_tripdata_2024-10.parquet: 100%|██████████| 61.4M/61.4M [00:00<00:00, 66.1MB/s]


Đang tải yellow_tripdata_2024-11.parquet...


yellow_tripdata_2024-11.parquet: 100%|██████████| 57.8M/57.8M [00:01<00:00, 48.9MB/s]


Đang tải yellow_tripdata_2024-12.parquet...


yellow_tripdata_2024-12.parquet: 100%|██████████| 58.7M/58.7M [00:01<00:00, 36.4MB/s]


# Crawl and process weather

In [48]:
import pandas as pd
import os
import requests
from sklearn.preprocessing import MinMaxScaler

def crawl_and_process_weather(years, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for year in years:
        print(f"\nĐang tải và xử lý dữ liệu thời tiết NYC năm {year}...")

        url = (
            f"https://archive-api.open-meteo.com/v1/archive?"
            f"latitude=40.7128&longitude=-74.0060&"
            f"start_date={year}-01-01&end_date={year}-12-31&"
            f"hourly=temperature_2m,precipitation,cloudcover,windspeed_10m&"
            f"timezone=America%2FNew_York"
        )

        response = requests.get(url).json()

        df = pd.DataFrame(response['hourly'])
        df['time'] = pd.to_datetime(df['time'])
        df.set_index('time', inplace=True)

        target_columns = ['temperature_2m', 'precipitation', 'cloudcover', 'windspeed_10m']
        df[target_columns] = df[target_columns].apply(pd.to_numeric, errors='coerce')
        df = df.resample('30min').interpolate(method='linear')

        weather_df = df.copy()

        scaler = MinMaxScaler()
        continuous_cols = ['temperature_2m', 'cloudcover', 'windspeed_10m']
        weather_df[continuous_cols] = scaler.fit_transform(weather_df[continuous_cols])

        weather_df['Rain_None'] = (weather_df['precipitation'] == 0.0).astype(int)
        weather_df['Rain_Light'] = ((weather_df['precipitation'] > 0.0) & (weather_df['precipitation'] <= 2.5)).astype(int)
        weather_df['Rain_Moderate'] = ((weather_df['precipitation'] > 2.5) & (weather_df['precipitation'] <= 7.6)).astype(int)
        weather_df['Rain_Heavy'] = (weather_df['precipitation'] > 7.6).astype(int)
        weather_df.drop(columns=['precipitation'], inplace=True)

        output_path = os.path.join(output_dir, f'weather_{year}_processed.csv')
        weather_df.to_csv(output_path)
        print(f"-> Đã lưu: {output_path}")

if __name__ == "__main__":
    TARGET_YEARS = range(2022, 2025)
    OUTPUT_DIR = "/content/drive/MyDrive/TLC_project/data/context"

    crawl_and_process_weather(TARGET_YEARS, OUTPUT_DIR)


Đang tải và xử lý dữ liệu thời tiết NYC năm 2022...
-> Đã lưu: /content/drive/MyDrive/TLC_project/data/context/weather_2022_processed.csv

Đang tải và xử lý dữ liệu thời tiết NYC năm 2023...
-> Đã lưu: /content/drive/MyDrive/TLC_project/data/context/weather_2023_processed.csv

Đang tải và xử lý dữ liệu thời tiết NYC năm 2024...
-> Đã lưu: /content/drive/MyDrive/TLC_project/data/context/weather_2024_processed.csv


# Generate holidays

In [46]:
!pip install holidays python-dateutil

In [49]:
import pandas as pd
import os
import holidays
from dateutil.easter import easter
from dateutil.relativedelta import relativedelta, SU

def process_dynamic_holidays(years, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for year in years:
        print(f"\nĐang khởi tạo ma trận Holidays cho năm {year}...")

        time_index = pd.date_range(start=f'{year}-01-01 00:00:00',
                                   end=f'{year}-12-31 23:30:00',
                                   freq='30min')
        holidays_df = pd.DataFrame(index=time_index)

        ny_holidays = holidays.US(state='NY', years=year)
        legal_holidays = [date.strftime('%Y-%m-%d') for date in ny_holidays.keys()]

        fixed_events = [f'{year}-02-14', f'{year}-10-31', f'{year}-12-24', f'{year}-12-31']
        easter_date = easter(year)
        mothers_day = pd.to_datetime(f'{year}-05-01') + relativedelta(weekday=SU(+2))
        fathers_day = pd.to_datetime(f'{year}-06-01') + relativedelta(weekday=SU(+3))

        event_festivals = fixed_events + [
            easter_date.strftime('%Y-%m-%d'),
            mothers_day.strftime('%Y-%m-%d'),
            fathers_day.strftime('%Y-%m-%d')
        ]

        def get_date_range(start, end, y=year):
            return pd.date_range(start=f'{y}-{start}', end=f'{y}-{end}').strftime('%Y-%m-%d').tolist()

        easter_str = easter_date.strftime('%Y-%m-%d')
        spring_break_start = (easter_date - pd.Timedelta(days=5)).strftime('%m-%d')
        spring_break_end = (easter_date + pd.Timedelta(days=2)).strftime('%m-%d')

        school_recess = (
            get_date_range('02-18', '02-23') +
            get_date_range(spring_break_start, spring_break_end) +
            get_date_range('12-24', '12-31')
        )

        date_strs = holidays_df.index.strftime('%Y-%m-%d')

        holidays_df['Is_Legal_Holiday'] = date_strs.isin(legal_holidays).astype(int)
        holidays_df['Is_School_Recess'] = date_strs.isin(school_recess).astype(int)
        holidays_df['Is_Event_Festival'] = date_strs.isin(event_festivals).astype(int)

        output_path = os.path.join(output_dir, f'holidays_{year}_processed.csv')
        holidays_df.to_csv(output_path)
        print(f"-> Đã lưu lịch thông minh tại: {output_path}")

if __name__ == "__main__":
    TARGET_YEARS = range(2022, 2025)
    OUTPUT_DIR = "/content/drive/MyDrive/TLC_project/data/context"

    process_dynamic_holidays(TARGET_YEARS, OUTPUT_DIR)


Đang khởi tạo ma trận Holidays cho năm 2022...
-> Đã lưu lịch thông minh tại: /content/drive/MyDrive/TLC_project/data/context/holidays_2022_processed.csv

Đang khởi tạo ma trận Holidays cho năm 2023...
-> Đã lưu lịch thông minh tại: /content/drive/MyDrive/TLC_project/data/context/holidays_2023_processed.csv

Đang khởi tạo ma trận Holidays cho năm 2024...
-> Đã lưu lịch thông minh tại: /content/drive/MyDrive/TLC_project/data/context/holidays_2024_processed.csv


In [52]:
!dvc add data/raw/taxi
!dvc add data/context

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
  3% 1.00/36.0 [00:00<00:14, 2.36file/s{'info': ''}]
  6% 2.00/36.0 [00:00<00:16, 2.07file/s{'info': ''}]
  8% 3.00/36.0 [00:01<00:17, 1.91file/s{'info': ''}]
 11% 4.00/36.0 [00:01<00:15, 2.13file/s{'info': ''}]
 14% 5.00/36.0 [00:02<00:15, 2.04file/s{'info': ''}]
 17% 6.00/36.0 [00:02<00:15, 1.98file/s{'info': ''}]
 19% 7.00/36.0 [00:03<00:13, 2.14file/s{'info': ''}]
 22% 8.00/36.0 [00:03<00:12, 2.21file/s{'info': ''}]
 25% 9.00/36.0 [00:04<00:10, 2.50file/s{'info': ''}]
 28% 10.0/36.0 [00:04<00:09, 2.62file/s{'info': ''}]
 31% 11.0/36.0 [00:04<00:08, 2.83file/s{'info': ''}]
 33% 12.0/36.0 [00:04<00:08, 2.96file/s{'info': ''}]
 36% 13.0/36.0 [00:05<00:07, 3.15file/s{'info': ''}]
 39% 14.0/36.0 [00:05<00:06, 3.21file/s{'info': ''}]
 42% 15.0/36.0 [00:05<00:06, 3.19file/s{'info': ''}]
 44% 16.0/36.0 [00:06<00:06, 3.19file/s{'info': ''}]
 47% 17.0/36.0 [00:06<00:06, 3.07file/s{'info

In [53]:
!git add data/raw/taxi.dvc data/context.dvc
!git add src/

In [54]:
!git commit -m "Thêm dữ liệu Taxi, Weather và Holidays mới"

[feature/data-update a79460a] Thêm dữ liệu Taxi, Weather và Holidays mới
 8 files changed, 197 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
 create mode 100644 data/context.dvc
 create mode 100644 data/raw/taxi.dvc
 create mode 100644 src/1_download_taxi_yellow.py
 create mode 100644 src/2_crawl_and_process_weather.py
 create mode 100644 src/3_generate_holidays_dynamic.py


In [59]:
!git add .dvc/config
!git commit -m "Đổi cấu hình DVC remote sang local path để chạy mượt trên Colab"

[feature/data-update e7c7571] Đổi cấu hình DVC remote sang local path để chạy mượt trên Colab
 1 file changed, 4 insertions(+)


In [60]:
!dvc push

Pushing
Querying remote cache:   0% 0/2 [00:00<?, ?files/s]
Querying remote cache:   0% 0/2 [00:00<?, ?files/s{'info': ''}]
                                                               
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Pushing to local:   0% 0/44 [00:00<?, ?file/s]
Pushing to local:   0% 0/44 [00:00<?, ?file/s{'info': ''}]
Pushing to local:  17% 1/6 [00:00<00:00,  7.95file/s{'info': ''}]
Pushing to local:  67% 4/6 [00:00<00:00, 17.93file/s{'info': ''}]
  0%|          |Pushing to local                  6/? [00:00<00:00, 17.07file/s]
Pushing to local:  22% 8/36 [00:01<00:06,  4.13file/s{'info': ''}]              
Pushing to local:  28% 10/36 [00:03<00:13,  1.98file/s{'info': ''}]
Pushing to local:  31% 11/36 [00:04<00:16,  1.56file/s{'info': ''}]
Pushing to local:  33% 12/36 [00:05<00:17,  1.41file/s{'info': ''}]
Pushing to local:  36% 13/36

In [61]:
!git status

On branch feature/data-update
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	DVC_and_Storage.ipynb
	data/.gitignore
	data/raw/.gitignore

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
!git push -u origin feature/data-update